# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [1]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [17]:
metricas = [
    'Spotify Streams',
    'Spotify Playlist Reach'
]

for columna in metricas:
    spotify_raw[columna] = pd.to_numeric(
        spotify_raw[columna].str.replace(',', '', regex=False),
        errors='coerce'
    )

In [33]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = np.arange(1, len(dim_tiempo) + 1)
fecha = pd.to_datetime(
    dim_tiempo['Release Date'],
    errors='coerce'
)

dim_tiempo['Year'] = fecha.dt.year
dim_tiempo['Month'] = fecha.dt.month
display(dim_tiempo)

,Release Date,id_tiempo,Year,Month
0,4/26/2024,1,2024,4
1,5/4/2024,2,2024,5
2,3/19/2024,3,2024,3
3,1/12/2023,4,2023,1
4,5/31/2024,5,2024,5
...,...,...,...,...
1557,4/9/2016,1558,2016,4
1558,6/5/2023,1559,2023,6
1559,10/31/2018,1560,2018,10
1560,11/8/2016,1561,2016,11


In [31]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_pista'] = np.arange(1, len(dim_pista) + 1)
display(dim_pista)

,Track,Artist,Explicit Track,id_pista
0,MILLION DOLLAR BABY,Tommy Richman,0,1
1,Not Like Us,Kendrick Lamar,1,2
2,i like the way you kiss me,Artemas,0,3
3,Flowers,Miley Cyrus,0,4
4,Houdini,Eminem,1,5
...,...,...,...,...
4531,For the Last Time,$uicideboy$,1,4532
4532,Dil Meri Na Sune,Atif Aslam,0,4533
4533,Grace (feat. 42 Dugg),Lil Baby,1,4534
4534,Nashe Si Chadh Gayi,Arijit Singh,0,4535


In [40]:
# 3. Crear fact_streams
df_temp = spotify_raw.merge(dim_tiempo, on=['Release Date'], how='left')
df_temp = df_temp.merge(dim_pista, on=['Track', 'Artist', 'Explicit Track' ], how='left')
fact_streams = df_temp[['id_pista', 'id_tiempo', 'Track Score', 'YouTube Views','Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach']]

print(fact_streams.head(3))

   id_pista  id_tiempo  Track Score YouTube Views  Spotify Streams  \
0         1          1        725.4    84,274,754      390470936.0   
1         2          2        545.9   116,347,040      323703884.0   
2         3          3        538.4   122,599,116      601309283.0   

  Spotify Playlist Count  Spotify Playlist Reach  
0                 30,716             196631588.0  
1                 28,113             174597137.0  
2                 54,331             211607669.0  


In [43]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
# TRANSFORM: limpiar las métricas con separadores de miles
    
cubo_olap = fact_streams.merge(dim_pista[['id_pista', 'Artist', 'Explicit Track', 'Track']], on='id_pista', how='left').merge(dim_tiempo[['id_tiempo', 'Year', 'Month']], on='id_tiempo', how='left')
analisis_artista= cubo_olap.groupby('Artist').agg(Streams=('Spotify Streams', 'sum'), Reach=('Spotify Playlist Reach', 'sum'), Score=('Track Score', 'mean'))
analisis_artista = analisis_artista.sort_values('Streams', ascending=False)

display(analisis_artista)

,Streams,Reach,Score
Artist,,,
Bad Bunny,3.705483e+10,1.465487e+09,47.300000
The Weeknd,3.694854e+10,1.302749e+09,48.690323
Drake,3.496216e+10,1.706306e+09,45.580952
Taylor Swift,3.447077e+10,2.242162e+09,40.900000
Post Malone,2.613747e+10,1.042300e+09,72.218182
...,...,...,...
3 Palavrinhas,0.000000e+00,0.000000e+00,142.400000
Leo Walrus,0.000000e+00,2.830000e+02,33.100000
Los Del Rio,0.000000e+00,0.000000e+00,21.400000


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [38]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: [Escribe aquí si es Roll-up, Drill-down, Slice o Dice]
respuesta_1 = cubo_olap.groupby('Artist').agg(Streams=('Spotify Streams', 'sum')).sort_values('Streams', ascending=False)
display(respuesta_1)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 2----')
respuesta_2 = cubo_olap[cubo_olap['Year'] == 2024]
display(respuesta_2)

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 3----')
respuesta_3 = cubo_olap.groupby(['Year', 'Month']).agg(Streams=('Spotify Streams', 'sum')).sort_index()
display(respuesta_3)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 4----')
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == True) & (cubo_olap['Year'] == 2024)]
display(respuesta_4)

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 5----')
respuesta_5 = cubo_olap.groupby('Year').agg(Average_Score=('Track Score', 'mean')).sort_index()
display(respuesta_5)

,Streams
Artist,
Bad Bunny,3.705483e+10
The Weeknd,3.694854e+10
Drake,3.496216e+10
Taylor Swift,3.447077e+10
Post Malone,2.613747e+10
...,...
3 Palavrinhas,0.000000e+00
Leo Walrus,0.000000e+00
Los Del Rio,0.000000e+00


---- RESPUESTA 2----


,id_pista,id_tiempo,Track Score,YouTube Views,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Artist,Explicit Track,Year,Month
0,1,1,725.4,"84,274,754",390470936.0,"30,716",196631588.0,Tommy Richman,0,2024,4
1,2,2,545.9,"116,347,040",323703884.0,"28,113",174597137.0,Kendrick Lamar,1,2024,5
2,3,3,538.4,"122,599,116",601309283.0,"54,331",211607669.0,Artemas,0,2024,3
4,5,5,423.3,"77,373,957",107034922.0,"7,223",151469874.0,Eminem,1,2024,5
6,7,7,407.2,"308,723,145",900158751.0,"73,118",201585714.0,Benson Boone,0,2024,1
...,...,...,...,...,...,...,...,...,...,...,...
4544,4481,361,19.6,"1,090,122",6184259.0,"1,641",8841671.0,Nemzzz,1,2024,5
4552,4489,13,19.6,"408,201",3970523.0,"1,230",392098.0,ian,1,2024,5
4561,4498,190,19.5,"9,700,826",1633647.0,160,2592948.0,Diljit Dosanjh,0,2024,5
4593,4530,616,19.4,"34,315,313",3754914.0,101,1306604.0,Thaman S,0,2024,3


---- RESPUESTA 3----


Streams
Year Month              
1987 7      1.879386e+09
1991 9      2.021910e+09
1994 1      1.810650e+09
1998 4      1.783399e+08
1999 6      1.405354e+09
...                  ...
2024 2      1.023077e+10
     3      1.263673e+10
     4      2.152035e+10
     5      1.005348e+10
     6      2.711085e+09

[191 rows x 1 columns]

---- RESPUESTA 4----


,id_pista,id_tiempo,Track Score,YouTube Views,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Artist,Explicit Track,Year,Month
1,2,2,545.9,"116,347,040",3.237039e+08,"28,113",174597137.0,Kendrick Lamar,1,2024,5
4,5,5,423.3,"77,373,957",1.070349e+08,"7,223",151469874.0,Eminem,1,2024,5
7,8,8,375.8,"228,382,568",6.750792e+08,"40,094",211236940.0,FloyyMenor,1,2024,2
8,9,9,355.7,NaN,1.653018e+09,1,15.0,MUSIC LAB JPN,1,2024,6
9,10,10,330.6,"32,735,244",9.067657e+07,"10,400",184199419.0,Central Cee,1,2024,5
...,...,...,...,...,...,...,...,...,...,...,...
4486,4425,123,19.7,"2,127,850",1.594834e+07,"4,948",2021225.0,J. Cole,1,2024,4
4491,4430,132,19.7,"2,292,547",9.126485e+06,"2,054",1013145.0,Lil Tecca,1,2024,5
4544,4481,361,19.6,"1,090,122",6.184259e+06,"1,641",8841671.0,Nemzzz,1,2024,5
4552,4489,13,19.6,"408,201",3.970523e+06,"1,230",392098.0,ian,1,2024,5


---- RESPUESTA 5----


,Average_Score
Year,
1987,21.500000
1991,19.600000
1994,31.500000
1998,20.800000
1999,22.200000
2000,30.400000
2001,20.100000
2002,26.060000
2003,23.766667


In [45]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 6----')
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
display(respuesta_6)

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 7 ----')
respuesta_7 = cubo_olap.groupby(['Artist', 'Track']).agg(Youtube=('YouTube Views', 'sum')).sort_index()
display(respuesta_7)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 8 ----')
respuesta_8 = cubo_olap[
    (cubo_olap['Month'] == 12) &
    (cubo_olap['Year'] == 2023)
]
display(respuesta_8)

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 9 ----')
respuesta_9 = cubo_olap.groupby('Month').agg(Total_Streams=('Spotify Streams', 'sum')).sort_values('Total_Streams', ascending=False)
display(respuesta_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: [Escribe aquí...]
print('---- RESPUESTA 10 ----')
respuesta_10 = cubo_olap[
    (cubo_olap['Artist'] == 'Taylor Swift') &
    (cubo_olap['Explicit Track'] == False) &
    (cubo_olap['Year'] == 2024)
]
display(respuesta_10)

---- RESPUESTA 6----


,id_pista,id_tiempo,Track Score,YouTube Views,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Artist,Explicit Track,Track,Year,Month
63,61,57,184.8,"1,516,863,120",1.811991e+09,"173,358",67603671.0,Bad Bunny,1,Me Porto Bonito,2022,5
88,86,57,163.1,"1,366,181,141",1.514240e+09,"190,692",77283361.0,Bad Bunny,1,Titï¿½ï¿½ Me Pregu,2022,5
102,100,91,150.7,"113,379,734",4.781509e+08,"73,306",51842133.0,Bad Bunny,1,MONACO,2023,10
174,171,91,114.2,"183,381,482",6.891815e+08,"61,138",90492257.0,Bad Bunny,1,PERRO NEGRO,2023,10
187,184,57,110.7,"945,857,087",1.421763e+09,"152,592",62313059.0,Bad Bunny,0,Ojitos Lindos,2022,5
245,242,204,98.4,"217,687,654",7.178490e+08,"98,440",37797217.0,Bad Bunny,1,WHERE SHE GOES,2023,5
261,258,135,96.1,"1,861,597,184",1.977947e+09,"245,685",56430031.0,Bad Bunny,1,Dakiti,2020,1
271,268,57,94.1,"765,780,154",1.108032e+09,"121,256",44739826.0,Bad Bunny,1,Moscow Mule,2022,5
290,287,57,90.1,"501,908,214",1.293774e+09,"115,332",51081689.0,Bad Bunny,1,Efecto,2022,5
294,291,238,88.6,"1,084,871,745",1.398634e+09,"177,082",44848091.0,Bad Bunny,1,Yonaguni,2021,6


---- RESPUESTA 7 ----


Youtube
Artist                                      Track                                                          
"XY"                                        Come and Say - Original Mix                         415,159,148
$OHO BANI                                   ZEIT, DASS SICH WAS DREHT                             6,744,201
$uicideboy$                                 ...And to Those I Love, Thanks for Sticking Around  269,460,083
                                            1000 Blunts                                          87,680,985
                                            Are You Going to See the Rose in the Vase, or t...    2,640,169
...                                                                                                     ...
ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý             ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý           44,586,424
ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý)            ýýýýýýýýýýýýýýýýýý                                    1,356,336
ýýýýýýýýýýýýýýýýýý                          ýýýýýýýýýýýýýýýýýýýýýýýýýýý                         127,116,431
ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý                      247,148,544
ýýýýýýýýýýýýýýýýýýýýýýýý                    ýýýýýýýýýýýýýýýýýýýýýýýý                             72,510,719

[4483 rows x 1 columns]

---- RESPUESTA 8 ----


,id_pista,id_tiempo,Track Score,YouTube Views,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Artist,Explicit Track,Track,Year,Month
61,59,55,186.5,"222,076",1384.0,2,801.0,mitsu sound,0,Very Cute Melody by Marimba Tone (39813),2023,12
94,92,84,155.9,"433,839,212",649404552.0,"27,544",47789063.0,Xavi,0,La Diabla,2023,12
97,95,87,153.1,"369,321,334",359803580.0,"26,065",79152427.0,Peso Pluma,1,BELLAKEO,2023,12
131,129,112,130.7,NaN,8327.0,3,8.0,WasteLand,0,Looking Like a Wow End of Year 2023,2023,12
177,174,152,112.6,"166,363,445",606124553.0,"40,628",139802900.0,Feid,0,LUNA,2023,12
...,...,...,...,...,...,...,...,...,...,...,...,...
4189,4133,979,20.7,"19,367,095",100634965.0,"8,790",29458776.0,Tiagz,1,Tacata (Remix),2023,12
4194,4138,87,20.7,"78,290,967",9129729.0,"1,422",3777844.0,NATTI NATASHA,0,YA NO TE EXTRAï¿½,2023,12
4385,4325,152,20.1,"8,478,416",45238166.0,"6,057",7158117.0,Beyoncï¿,1,MY HOUSE,2023,12
4462,4401,1533,19.8,"72,510,719",3225791.0,411,11826.0,ýýýýýýýýýýýýýýýýýýýýýýýý,0,ýýýýýýýýýýýýýýýýýýýýýýýý,2023,12


---- RESPUESTA 9 ----


,Total_Streams
Month,
1,3.652168e+11
5,1.779293e+11
11,1.734614e+11
10,1.672667e+11
3,1.638041e+11
7,1.503832e+11
6,1.484293e+11
4,1.462846e+11
8,1.426826e+11


---- RESPUESTA 10 ----


,id_pista,id_tiempo,Track Score,YouTube Views,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Artist,Explicit Track,Track,Year,Month
16,17,17,297.6,"104,673,836",395433400.0,"12,784",177932568.0,Taylor Swift,0,Fortnight (feat. Post Malone),2024,4
866,861,17,52.3,"16,423,009",166168092.0,"5,004",25607992.0,Taylor Swift,0,Whoï¿½ï¿½ï¿½s Afraid of Little O,2024,4
1013,1007,17,47.8,"15,237,337",175613480.0,"4,454",20883678.0,Taylor Swift,0,"So Long, London",2024,4
1167,1159,17,44.0,"13,509,432",161916445.0,"3,784",17036800.0,Taylor Swift,0,My Boy Only Breaks His Favorite Toys,2024,4
1270,1260,17,42.1,"9,873,283",165611453.0,"4,418",21928475.0,Taylor Swift,0,Guilty as Sin?,2024,4
2484,2454,183,28.6,"5,528,058",97601615.0,"3,397",13075013.0,Taylor Swift,0,So High School,2024,4
2517,2486,17,28.3,"5,521,347",106201248.0,"2,992",15108991.0,Taylor Swift,0,The Alchemy,2024,4
2528,2497,17,28.2,NaN,250430303.0,648,818594.0,Taylor Swift,0,I Can Do It With a Broken Heart,2024,4
2776,2744,183,26.7,"7,381,186",88244010.0,"2,659",8584946.0,Taylor Swift,0,imgonnagetyouback,2024,4
2816,2784,17,26.5,"7,227,379",105424914.0,"2,473",7785632.0,Taylor Swift,0,Fresh Out The Slammer,2024,4
